# Notebook 01: NHANES EDA
## T2D Risk Predictor — Fairness-Aware ML for Healthcare
**PhD Research | University of Portsmouth | Kesser Karim**

### Objectives
1. Load and inspect NHANES 2017-2020 data (demographics + biomarkers)
2. Understand missing data patterns across subgroups
3. Visualise T2D prevalence by ethnicity, sex, and age band
4. Identify class imbalance and distribution shifts across subgroups

### Data Source
NHANES 2017-2020 Pre-Pandemic | CDC | https://www.cdc.gov/nchs/nhanes/
Key files: DEMO_P.XPT, GLU_P.XPT, BMX_P.XPT, BPX_P.XPT, DIQ_P.XPT

In [ ]:
# === SETUP ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('../data/raw')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load NHANES Files

In [ ]:
# Load core NHANES tables (download from CDC first — see README)
demo = pd.read_sas(DATA_DIR / 'DEMO_P.XPT', format='xport', encoding='utf-8')
glu  = pd.read_sas(DATA_DIR / 'GLU_P.XPT',  format='xport', encoding='utf-8')
bmx  = pd.read_sas(DATA_DIR / 'BMX_P.XPT',  format='xport', encoding='utf-8')
diq  = pd.read_sas(DATA_DIR / 'DIQ_P.XPT',  format='xport', encoding='utf-8')
bpx  = pd.read_sas(DATA_DIR / 'BPX_P.XPT',  format='xport', encoding='utf-8')

# Merge on SEQN (participant ID)
df = demo.merge(glu, on='SEQN', how='left') \
         .merge(bmx, on='SEQN', how='left') \
         .merge(diq, on='SEQN', how='left') \
         .merge(bpx, on='SEQN', how='left')

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Define T2D Label & Subgroups

In [ ]:
# T2D label: fasting glucose >= 126 mg/dL OR diagnosed (DIQ010 == 1)
df['t2d'] = ((df['LBXGLU'] >= 126) | (df['DIQ010'] == 1)).astype(int)

# Ethnicity mapping (RIDRETH3)
eth_map = {1: 'Mexican American', 2: 'Other Hispanic', 3: 'Non-Hispanic White',
           4: 'Non-Hispanic Black', 6: 'Non-Hispanic Asian', 7: 'Other/Multi'}
df['ethnicity'] = df['RIDRETH3'].map(eth_map)

# Sex and age band
df['sex'] = df['RIAGENDR'].map({1: 'Male', 2: 'Female'})
df['age_band'] = pd.cut(df['RIDAGEYR'], bins=[0,40,55,70,120],
                        labels=['<40', '40-55', '55-70', '70+'])

print('T2D prevalence:', df['t2d'].mean().round(3))

## 3. Subgroup Prevalence Analysis (Core Fairness Insight)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# T2D prevalence by ethnicity
eth_prev = df.groupby('ethnicity')['t2d'].mean().sort_values(ascending=False)
eth_prev.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('T2D Prevalence by Ethnicity', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Prevalence')
axes[0].tick_params(axis='x', rotation=30)

# T2D prevalence by sex
sex_prev = df.groupby('sex')['t2d'].mean()
sex_prev.plot(kind='bar', ax=axes[1], color=['#E07B54', '#5B8DB8'], edgecolor='white')
axes[1].set_title('T2D Prevalence by Sex', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Prevalence')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_subgroup_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Data Assessment

Key question: are missingness patterns uniform across subgroups, or do certain populations have more sparse biomarker data?

In [ ]:
features = ['LBXGLU', 'BMXBMI', 'BPXOSY1', 'RIDAGEYR', 'INDFMPIR']
missing = df.groupby('ethnicity')[features].apply(lambda x: x.isnull().mean())

plt.figure(figsize=(10, 4))
sns.heatmap(missing.T, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Missing rate'})
plt.title('Missing Data Rate by Feature & Ethnicity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_missing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. EDA Summary & Next Steps

| Finding | Implication |
|---------|-------------|
| Non-Hispanic Asian populations have 2x T2D risk at lower BMI | Standard BMI thresholds underestimate risk |
| Fasting glucose data missing 40%+ for some subgroups | Need imputation strategy before modelling |
| Strong age-T2D correlation (>70: 40% prevalence) | Age band is key stratification variable |

**Next:** `02_baseline_model.ipynb` — XGBoost baseline with 5-fold CV, AUROC, and calibration curves